# Supremum Visualization Test

Testing the numerical supremum and ν-measure calculations with `plot_supremum`.


### IMPORTS

In [1]:
# Setup path and imports
import sys
from pathlib import Path

# Add AFIS_Repo root to path
REPO_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(REPO_ROOT))

# Import from afis package
from afis.core.afis_utils import Triangular, Gaussian, Trapezoidal, FuzzySet
from afis.visualization.plotting import (
    plot_supremum, create_rule_base, run_afis, plot_results, show_svi_table
)
from afis.core.A_FIS import format_FN_N_Dim


### A-FIS Inference with Gaussian Input (1D Case)


In [2]:
# Define antecedents (trapezoidal)
A_1 = Trapezoidal(ini=10, top1=20, top2=20, end=35)
A_2 = Trapezoidal(ini=40, top1=50, top2=65, end=70)
A_3 = Trapezoidal(ini=95, top1=110, top2=110, end=120)
A_4 = Trapezoidal(ini=120, top1=140, top2=140, end=140)

# Define consequents (trapezoidal)
B_1 = Trapezoidal(ini=0, top1=0, top2=0, end=100)
B_2 = Trapezoidal(ini=0, top1=100, top2=100, end=200)
B_3 = Trapezoidal(ini=100, top1=200, top2=200, end=300)
B_4 = Trapezoidal(ini=200, top1=300, top2=300, end=300)

# Create FuzzySet objects
A_1_fs, A_2_fs = FuzzySet("A_1", A_1), FuzzySet("A_2", A_2)
A_3_fs, A_4_fs = FuzzySet("A_3", A_3), FuzzySet("A_4", A_4)
B_1_fs, B_2_fs = FuzzySet("B_1", B_1), FuzzySet("B_2", B_2)
B_3_fs, B_4_fs = FuzzySet("B_3", B_3), FuzzySet("B_4", B_4)

# Create rule base
rule_base = create_rule_base(
    antecedents=[[A_1_fs, A_2_fs, A_3_fs, A_4_fs]],
    consequents=[B_1_fs, B_2_fs, B_3_fs, B_4_fs],
    input_ranges=[(0, 150)],
    output_range=(0, 300),
    rules=[([0], 0), ([1], 1), ([2], 2), ([3], 3)]
)
print("Rule base: A_1→B_1, A_2→B_2, A_3→B_3, A_4→B_4")


Rule base: A_1→B_1, A_2→B_2, A_3→B_3, A_4→B_4


In [3]:
# Define Gaussian input (centered near A_2 region)
input = Gaussian(center=75, sigma=8)
# input = Trapezoidal(ini=60, top1=75, top2=75, end=90)

input_list = [input]

# Format input for A_FIS
input_fuzzy_sets = [FuzzySet("X", input)]
input_formatted = format_FN_N_Dim(input_fuzzy_sets, ftype='rule_antecedent')

# Run A_FIS inference
output, U_out, y_crisp, max_values = run_afis(
    input_formatted, 
    rule_base,
    imp_params=['luka', 1], 
    t_norm_type='product'
)

# print(f"Gaussian Input")
print(f"Defuzzified output: {y_crisp:.3f}")

# Plot: antecedents + input
plot_results(rule_base, input_list, output, U_out, y_crisp)

# Show SVI table
show_svi_table(rule_base, input_formatted)


Defuzzified output: 116.066



Rule Base:
Rule       Antecedent           Consequent          
------------------------------------------------------------------------------------------
0          A_1                  B_1                 
1          A_2                  B_2                 
2          A_3                  B_3                 
3          A_4                  B_4                 

A-subsethood Measures:
Rule       SVI             A-subsethood (S_A)  
----------------------------------------------------------------------
0          0.616489        0.000000            
1          0.883154        0.686898            
2          0.749824        0.166628            
3          0.566390        0.000000            


### A-FIS Inference with Gaussian Inputs (3D Case)


In [4]:
# Define antecedents for 3 dimensions
# Dimension 1: A1, A2, A3
A1 = Trapezoidal(ini=0, top1=15, top2=15, end=30)
A2 = Trapezoidal(ini=40, top1=50, top2=50, end=60)
A3 = Trapezoidal(ini=80, top1=100, top2=100, end=100)

# Dimension 2: B1, B2
B1 = Trapezoidal(ini=0, top1=0, top2=0, end=60)
B2 = Trapezoidal(ini=130, top1=170, top2=200, end=200)

# Dimension 3: C1, C2, C3
C1 = Trapezoidal(ini=0, top1=0, top2=0, end=60)
C2 = Trapezoidal(ini=70, top1=80, top2=90, end=110)
C3 = Trapezoidal(ini=130, top1=150, top2=150, end=150)

# Consequents: D1, D2, D3
D1 = Trapezoidal(ini=20, top1=50, top2=50, end=70)
D2 = Trapezoidal(ini=120, top1=150, top2=180, end=200)
D3 = Trapezoidal(ini=230, top1=270, top2=300, end=300)

# Create rule base
rule_base_3d = create_rule_base(
    antecedents=[
        [FuzzySet("A1", A1), FuzzySet("A2", A2), FuzzySet("A3", A3)],
        [FuzzySet("B1", B1), FuzzySet("B2", B2)],
        [FuzzySet("C1", C1), FuzzySet("C2", C2), FuzzySet("C3", C3)]
    ],
    consequents=[FuzzySet("D1", D1), FuzzySet("D2", D2), FuzzySet("D3", D3)],
    input_ranges=[(0, 100), (0, 200), (0, 150)],
    output_range=(0, 300),
    rules=[
        ([0, 0, 0], 0),  # A1, B1, C1 → D1
        ([1, 0, 1], 1),  # A2, B1, C2 → D2
        ([2, 1, 2], 2),  # A3, B2, C3 → D3
        ([1, 0, 1], 2)   # A2, B1, C2 → D3
    ]
)
print("3D Rule base created: 4 rules over 3 dimensions")


3D Rule base created: 4 rules over 3 dimensions


In [5]:
# Define Gaussian inputs for each dimension
input_3d_1 = Gaussian(center=45, sigma=8)   # Near A2 region
input_3d_2 = Gaussian(center=50, sigma=15)  # Between B1 and B2
input_3d_3 = Gaussian(center=85, sigma=10)  # Near C2 region

input_list_3d = [input_3d_1, input_3d_2, input_3d_3]

# Format inputs for A_FIS
input_fuzzy_sets_3d = [FuzzySet(f"X{i+1}", inp) for i, inp in enumerate(input_list_3d)]
input_formatted_3d = format_FN_N_Dim(input_fuzzy_sets_3d, ftype='rule_antecedent')

# Run A_FIS inference
output_3d, U_3d, y_crisp_3d, _ = run_afis(
    input_formatted_3d, 
    rule_base_3d,
    imp_params=['luka', 1], 
    t_norm_type='product'
)

print(f"Gaussian Inputs: X1(μ={input_3d_1.center}, σ={input_3d_1.sigma}), "
      f"X2(μ={input_3d_2.center}, σ={input_3d_2.sigma}), "
      f"X3(μ={input_3d_3.center}, σ={input_3d_3.sigma})")
print(f"Defuzzified output: {y_crisp_3d:.3f}")

# Plot antecedents + inputs for each dimension, and consequents + output
plot_results(rule_base_3d, input_list_3d, output_3d, U_3d, y_crisp_3d)

# Show SVI table
show_svi_table(rule_base_3d, input_formatted_3d)


Gaussian Inputs: X1(μ=45, σ=8), X2(μ=50, σ=15), X3(μ=85, σ=10)
Defuzzified output: 205.729



Rule Base:
Rule       Antecedents                                        Consequent          
------------------------------------------------------------------------------------------
0          A1, B1, C1                                         D1                  
1          A2, B1, C2                                         D2                  
2          A3, B2, C3                                         D3                  
3          A2, B1, C2                                         D3                  

A-subsethood Measures:
Rule       S_A(dim1)     S_A(dim2)     S_A(dim3)     S_A (agg)      
-------------------------------------------------------------------
0          0.096391      0.568942      0.000000      0.221778       
1          0.763507      0.568942      0.979671      0.770707       
2          0.000000      0.119436      0.000000      0.039812       
3          0.763507      0.568942      0.979671      0.770707       
